# Verified whole-board tactical curriculum

Purpose: generate thousands of positive and negative action labels for explicit capture, escape, connection and three-ply forced-capture obligations. The labels come from independently checked Go rules and finite predicates. No external engine teaches moves.

Inputs: original rule/verifier exports and a deterministic legal trajectory generator. Outputs: `data/curriculum.json`, its hash manifest, and reports under `reports/ahr_native_v5/`. Expected generation runtime: approximately 20–90 minutes, depending on forced-capture coverage; the default Run All validates existing data only and does not silently launch a long generation.

Checkpoint: `checkpoints/ahr-v5-curriculum-generation/`, saved at least every four trajectories or ten minutes. Exact resume: call `v5_generate_curriculum(project, V5CurriculumConfig())` with the same frozen definitions and configuration. Source-game and canonical local-family splits are assigned before D4 augmentation. UNKNOWN cutoffs never become failure labels.

In [ ]:
from pathlib import Path
import nbformat
project=Path.cwd().resolve()
while not (project/'notebooks/00_instance_setup_and_recovery.ipynb').exists():project=project.parent
for cell in nbformat.read(project/'experiments/ahr_native_v5/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,globals())
v5_load_v4(project,globals())

print('Shared Go rules, independent verifier, and native AHR definitions loaded.')

In [ ]:
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
import copy, hashlib, json, random, time

V5_CURRICULUM_SCHEMA='ahr_native_verified_curriculum_v5'
V5_CURRICULUM_PATH='experiments/ahr_native_v5/data/curriculum.json'
V5_CURRICULUM_NOTEBOOK='experiments/ahr_native_v5/02_verified_curriculum.ipynb'
V5_GOALS=('capture_one_ply','survive_two_plies','connect_one_ply','force_capture_three_plies')

@dataclass(frozen=True)
class V5CurriculumConfig:
    seed:int=2026100205
    train_per_goal:int=600
    development_per_goal:int=250
    max_games:int=3000
    plies_per_game:int=100
    sample_every:int=3
    max_tasks_per_game:int=24
    max_per_goal_per_game:int=8
    force_reply_budget:int=82
    purpose:str='verified_curriculum'
    def __post_init__(self):
        if min(self.train_per_goal,self.development_per_goal,self.max_games,self.plies_per_game,self.sample_every,self.max_tasks_per_game,self.max_per_goal_per_game,self.force_reply_budget)<1:raise ValueError('Positive curriculum bounds required')
        if self.force_reply_budget>82:raise ValueError('At most all 82 legal reply ids')

def v5_curriculum_family(state,targets,goal):
    # Canonical target neighbourhood, including the entire target extent and three-point halo.
    # D4 variants share a family; distant unrelated stones do not create new families.
    views=[]
    for symmetry in range(8):
        board=transform_go(state,symmetry);points=sorted(v4_action_transform(p,symmetry,9) for p in targets)
        rows=[p//9 for p in points];cols=[p%9 for p in points]
        top=max(0,min(rows)-3);bottom=min(9,max(rows)+4);left=max(0,min(cols)-3);right=min(9,max(cols)+4)
        patch=[[int(board.board[r*9+c]*state.to_play) for c in range(left,right)] for r in range(top,bottom)]
        mask=[[int(r*9+c in points) for c in range(left,right)] for r in range(top,bottom)]
        views.append(json.dumps([goal,top==0,bottom==9,left==0,right==9,patch,mask],separators=(',',':')))
    return hashlib.sha256(min(views).encode()).hexdigest()

def v5_curriculum_split(family):return 'development' if int(family[:8],16)%5==0 else 'train'

def v5_task_candidates(state):
    friendly=[]
    for stones,libs in groups_and_liberties(state.board,9):
        targets=sorted(stones);color=state.board[targets[0]]
        if color==state.to_play:
            friendly.append((set(stones),set(libs)))
            if len(libs)==1:yield 'survive_two_plies',targets,sorted(libs)
        elif len(libs)==1:yield 'capture_one_ply',targets,sorted(libs)
        elif len(libs)==2:yield 'force_capture_three_plies',targets,sorted(libs)
    seen=set()
    for i,(left,left_libs) in enumerate(friendly):
        for right,right_libs in friendly[i+1:]:
            shared=left_libs&right_libs
            if shared:
                targets=(min(left),min(right))
                if targets not in seen:seen.add(targets);yield 'connect_one_ply',list(targets),sorted(shared)

def v5_bounded_action(case,action,reply_budget=82):
    # Exact finite tactical predicates. A cutoff is UNKNOWN, never a negative label.
    state=from_dict(case['state']);child=arena_reference_child(state,int(action))
    if child is None:return {'status':'ILLEGAL','witness':None,'scope':'finite tactical predicate'}
    targets=case['targets'];goal=case['goal'];color=state.board[targets[0]]
    if goal=='capture_one_ply':
        good=all(child.board[p]!=color for p in targets)
        return {'status':'SAT' if good else 'UNSAT','witness':{'action':int(action),'surviving_targets':[p for p in targets if child.board[p]==color]}}
    if goal=='connect_one_ply':
        good=all(child.board[p]==state.to_play for p in targets)
        if good:good=set(targets)<=arena_reference_group(child.board,9,targets[0])[0]
        return {'status':'SAT' if good else 'UNSAT','witness':{'action':int(action),'connected':bool(good)}}
    surviving=[p for p in targets if child.board[p]==color]
    if goal=='survive_two_plies':
        if len(surviving)!=len(targets):return {'status':'UNSAT','witness':{'action':int(action),'loss_on_first_move':True}}
        stones,libs=arena_reference_group(child.board,9,surviving[0])
        # A Go move can remove an existing group only by filling its final liberty.
        if len(libs)>1:return {'status':'SAT','witness':{'action':int(action),'liberties_after':sorted(libs),'argument':'no one-move capture with two or more liberties'}}
        if not libs:raise RuntimeError('Rule verifier emitted a zero-liberty existing group')
        reply=next(iter(libs));after=arena_reference_child(child,reply)
        good=after is None or all(after.board[p]==color for p in targets)
        return {'status':'SAT' if good else 'UNSAT','witness':{'action':int(action),'critical_reply':reply,'critical_reply_legal':after is not None}}
    if goal!='force_capture_three_plies':raise ValueError('Unsupported tactical goal')
    if not surviving:return {'status':'SAT','witness':{'action':int(action),'already_captured':True}}
    if child.terminal:return {'status':'UNSAT','witness':{'action':int(action),'reason':'terminal_board_still_contains_target'}}
    _,libs=arena_reference_group(child.board,9,surviving[0])
    if len(libs)!=1:return {'status':'UNSAT','witness':{'action':int(action),'counterexample_reply':81,'remaining_liberties':sorted(libs)}}
    replies=[];visited=0
    for reply in range(82):
        after=arena_reference_child(child,reply)
        if after is None:continue
        if visited>=reply_budget:return {'status':'UNKNOWN','witness':None,'reason':'reply_budget','checked_replies':visited}
        visited+=1;remaining=[p for p in targets if after.board[p]==color]
        if not remaining:replies.append({'reply':reply,'already_captured':True});continue
        _,response_libs=arena_reference_group(after.board,9,remaining[0])
        if len(response_libs)!=1:return {'status':'UNSAT','witness':{'action':int(action),'counterexample_reply':reply,'remaining_liberties':sorted(response_libs)}}
        capture=next(iter(response_libs));last=arena_reference_child(after,capture)
        if last is None or any(last.board[p]==color for p in targets):return {'status':'UNSAT','witness':{'action':int(action),'counterexample_reply':reply,'capture':capture,'capture_legal':last is not None}}
        replies.append({'reply':reply,'capture':capture})
    return {'status':'SAT','witness':{'action':int(action),'all_legal_reply_continuations':replies}}

def v5_verified_task(state,goal,targets,reply_budget=82):
    case={'state':to_dict(state),'goal':goal,'targets':list(targets)};good=[];bad=[];unknown=[];evidence={}
    for action in legal_actions(state):
        result=v5_bounded_action(case,action,reply_budget)
        (good if result['status']=='SAT' else bad if result['status']=='UNSAT' else unknown).append(int(action))
        evidence[str(action)]=result
    if not good:return None,{'negative_actions':len(bad),'unknown_actions':len(unknown)}
    policy=[0.0]*82
    for action in good:policy[action]=1/len(good)
    family=v5_curriculum_family(state,targets,goal)
    row={'state':case['state'],'tactical_goal':goal,'tactical_targets':list(targets),'tactical_policy_target':policy,
         'tactical_success_actions':good,'tactical_failure_actions':bad,'tactical_unknown_actions':unknown,
         'policy_weight':0.0,'policy_target':[0.0]*82,'family_id':family,'split':v5_curriculum_split(family),
         'source':'verified_rules_and_ahr_task_curriculum','bounded_objective':{'kind':goal,'horizon':3 if 'three' in goal else 2 if 'two' in goal else 1,'status':'VERIFIED_POSITIVE_AND_NEGATIVE_ACTIONS','full_game_result':'UNKNOWN','cutoff_status':'UNKNOWN'},
         'verification':{'independent_reference_rules':True,'external_engine_labels':False,'action_evidence':evidence}}
    return row,{'negative_actions':len(bad),'unknown_actions':len(unknown)}

def v5_read_curriculum(project):
    project=Path(project);path=project/V5_CURRICULUM_PATH;manifest=path.with_suffix('.manifest.json')
    if not path.exists():raise FileNotFoundError('Generate the v5 verified curriculum first')
    descriptor=v5_read_json(manifest)
    if file_sha256(path)!=descriptor['sha256']:raise RuntimeError('Verified curriculum hash mismatch')
    data=v5_read_json(path)
    if data.get('schema')!=V5_CURRICULUM_SCHEMA:raise RuntimeError('Wrong curriculum schema')
    result={'train':[],'development':[]}
    for row in data['examples']:
        if row['split'] not in result:raise RuntimeError('Unexpected/final data in tactical curriculum')
        if v5_curriculum_split(row['family_id'])!=row['split']:raise RuntimeError('Canonical family split mismatch')
        result[row['split']].append(row)
    train_families={r['family_id'] for r in result['train']};dev_families={r['family_id'] for r in result['development']}
    train_games={r['provenance']['game_id'] for r in result['train']};dev_games={r['provenance']['game_id'] for r in result['development']}
    if train_families&dev_families or train_games&dev_games:raise RuntimeError('Tactical training/development leakage')
    result['provenance']=descriptor
    return result

v5_curriculum=v5_read_curriculum
v5_tactical_curriculum=v5_read_curriculum

def v5_generate_curriculum(project,config=None,stop_after_games=None,deadline_unix=None):
    project=Path(project);config=config if isinstance(config,V5CurriculumConfig) else V5CurriculumConfig(**(config or {}))
    path=project/V5_CURRICULUM_PATH;run_dir=project/'checkpoints/ahr-v5-curriculum-generation'
    identity={'schema':V5_CURRICULUM_SCHEMA,'config':asdict(config),'definition_sha256':v5_code_sha(project/V5_CURRICULUM_NOTEBOOK),'upstream':v4_source_identity(project)}
    if path.exists():
        descriptor=v5_read_json(path.with_suffix('.manifest.json'))
        if descriptor['identity']!=identity:raise RuntimeError('Existing curriculum has different source/config; refusing overwrite')
        v5_read_curriculum(project);return descriptor
    with checkpoint_lock(run_dir):
        saved=None;manifest_path=run_dir/'manifest.json'
        if manifest_path.exists():
            manifest=v5_read_json(manifest_path)
            for entry in manifest.get('entries',[]):
                payload=_checkpoint_entry_state(run_dir,entry)
                if payload is None or payload.get('identity')!=identity:raise RuntimeError('Curriculum checkpoint/source/config corrupt or changed')
                if entry['file']==manifest.get('latest'):saved=payload
            if saved is None:raise RuntimeError('Invalid latest curriculum checkpoint')
        elif list(run_dir.glob('*.pt')):raise RuntimeError('Curriculum payload without manifest')
        if saved is None:saved={'schema':V5_CURRICULUM_SCHEMA,'identity':identity,'config':asdict(config),'global_step':0,'next_game':0,'model':None,'optimizer':None,'scheduler':None,'scaler':None,'rng':{},'sampler':{'position':0},'metrics':{},'examples':[],'counts':{},'unknown':{}}
        last_committed_step=saved['global_step'] if manifest_path.exists() else -1
        examples=saved['examples'];families={r['family_id'] for r in examples};counts=Counter((r['split'],r['tactical_goal']) for r in examples);unknown=Counter(saved['unknown']);start=saved['next_game'];last_save=time.monotonic()
        quotas={'train':config.train_per_goal,'development':config.development_per_goal}
        for game_id in range(start,config.max_games):
            if stop_after_games is not None and game_id-start>=stop_after_games:break
            if deadline_unix is not None and time.time()>=deadline_unix:break
            rng=random.Random(config.seed+1000003*game_id);game_split='development' if game_id%5==0 else 'train';board=initial_state(9,7.5);local=Counter();accepted=0;actions=[]
            for ply in range(config.plies_per_game):
                legal=[a for a in legal_actions(board) if a!=81]
                if not legal:break
                # Bias to nearby liberties without using another Go engine or label source.
                near=set()
                for stones,libs in groups_and_liberties(board.board,9):
                    if len(libs)<=3:near.update(libs)
                pool=[a for a in legal if a in near]
                action=rng.choice(pool if pool and rng.random()<.6 else legal);child=play(board,action)
                if not verify_go_emission(board,action,child):raise RuntimeError('Curriculum legal trajectory failed independent verifier')
                board=child;actions.append(action)
                if ply<16 or ply%config.sample_every or accepted>=config.max_tasks_per_game:continue
                for goal,targets,_ in v5_task_candidates(board):
                    if counts[(game_split,goal)]>=quotas[game_split] or local[goal]>=config.max_per_goal_per_game or accepted>=config.max_tasks_per_game:continue
                    family=v5_curriculum_family(board,targets,goal)
                    if family in families or v5_curriculum_split(family)!=game_split:continue
                    row,stats=v5_verified_task(board,goal,targets,config.force_reply_budget);unknown[goal]+=stats['unknown_actions']
                    if row is None:continue
                    row['provenance']={'seed':config.seed,'game_id':game_id,'ply':len(actions),'prefix_actions':list(actions),'source_game_split':game_split,'generator':'verified_legal_whole_board_trajectory','external_engine_labels':False}
                    examples.append(row);families.add(family);counts[(game_split,goal)]+=1;local[goal]+=1;accepted+=1
            saved.update(global_step=game_id+1,next_game=game_id+1,examples=examples,counts={s+':'+g:n for (s,g),n in counts.items()},unknown=dict(unknown),rng={'per_game_seed':config.seed+1000003*(game_id+1)},sampler={'position':game_id+1})
            if game_id%4==0 or time.monotonic()-last_save>=600:
                _save_checkpoint_unlocked(run_dir,saved,best=True,resume_cell=V5_CURRICULUM_NOTEBOOK+': v5_generate_curriculum(project, identical config)');last_save=time.monotonic();last_committed_step=saved['global_step']
                v5_atomic_json(v5_report_path(project,'curriculum_progress.json'),{'status':'RUNNING','next_game':saved['next_game'],'examples':len(examples),'counts':saved['counts'],'unknown_actions':dict(unknown),'updated_unix':time.time()})
            if all(counts[(split,goal)]>=quota for split,quota in quotas.items() for goal in V5_GOALS):break
        if saved['global_step']!=last_committed_step:
            _save_checkpoint_unlocked(run_dir,saved,best=True,resume_cell=V5_CURRICULUM_NOTEBOOK+': v5_generate_curriculum(project, identical config)')
        complete=all(counts[(split,goal)]>=quota for split,quota in quotas.items() for goal in V5_GOALS)
        if not complete:
            receipt={'status':'INCOMPLETE','next_game':saved['next_game'],'counts':saved['counts'],'examples':len(examples),'identity':identity,'unknown_actions':dict(unknown)}
            v5_atomic_json(v5_report_path(project,'curriculum_progress.json'),receipt);return receipt
        data={'schema':V5_CURRICULUM_SCHEMA,'identity':identity,'examples':examples,'claims':'Exact bounded tactical success and failure only; full game outcome UNKNOWN; all cutoff actions excluded from negative labels','unknown_actions':dict(unknown)}
        v5_atomic_json(path,data)
        descriptor={'status':'COMPLETE','path':str(path.relative_to(project)),'sha256':file_sha256(path),'identity':identity,'counts':saved['counts'],'examples':len(examples),'source_games':len({r['provenance']['game_id'] for r in examples}),'family_count':len(families),'lockbox_used':False,'external_engine_labels':False,'unknown_actions':dict(unknown),'split_policy':'Source games and D4 canonical target neighbourhood families separated before augmentation'}
        v5_atomic_json(path.with_suffix('.manifest.json'),descriptor);v5_atomic_json(v5_report_path(project,'curriculum_progress.json'),descriptor)
        v5_read_curriculum(project);return descriptor

def v5_curriculum_checks(project):
    data=v5_read_curriculum(project);checks={}
    for split in ('train','development'):
        rows=data[split];checks[split+'_examples']=len(rows);checks[split+'_goals']=dict(Counter(r['tactical_goal'] for r in rows))
        checks[split+'_negative_actions']=sum(len(r['tactical_failure_actions']) for r in rows)
        for row in rows[::max(1,len(rows)//20)]:
            board=from_dict(row['state'])
            replay=initial_state(9,7.5)
            for action in row['provenance']['prefix_actions']:replay=play(replay,action)
            if to_dict(replay)!=row['state']:raise RuntimeError('Whole-board task chronology invalid')
            case={'state':row['state'],'goal':row['tactical_goal'],'targets':row['tactical_targets']}
            for action in row['tactical_success_actions']:
                if v5_bounded_action(case,action)['status']!='SAT':raise RuntimeError('Positive tactical label invalid')
            for action in row['tactical_failure_actions'][:3]:
                if v5_bounded_action(case,action)['status']!='UNSAT':raise RuntimeError('Negative tactical label invalid')
    receipt={'status':'PASS','checks':checks,'provenance':data['provenance'],'claim':'bounded verified action labels and strict split separation, no playing-strength claim'}
    v5_atomic_json(v5_report_path(project,'curriculum_validation.json'),receipt);return receipt


def v5_validate_terminal_capture_guard(project):
    legacy=v4_tactical_curriculum(project)
    row=next(row for row in legacy['train'] if row['tactical_goal']=='capture_one_ply')
    state=copy.deepcopy(row['state']);state['consecutive_passes']=1
    case={'state':state,'goal':'force_capture_three_plies','targets':row['tactical_targets']}
    result=v5_bounded_action(case,81);board=from_dict(state);child=play(board,81)
    checks={'second_pass_is_terminal':child.terminal,'target_survives_terminal':any(child.board[p]==board.board[p] for p in case['targets']),'finite_capture_false_is_unsat':result['status']=='UNSAT','explicit_terminal_reason':result['witness'].get('reason')=='terminal_board_still_contains_target'}
    if not all(checks.values()):raise RuntimeError('Terminal finite-capture guard failed: '+str(checks))
    receipt={'status':'PASS','checks':checks,'definition_sha256':v5_code_sha(Path(project)/V5_CURRICULUM_NOTEBOOK),'scope':'Terminal second pass cannot prove capture of a still-present enemy target'}
    v5_atomic_json(v5_report_path(project,'terminal_capture_guard.json'),receipt);return receipt


In [ ]:
import os, pandas as pd
from IPython.display import display
if os.environ.get('AHRV5_GENERATE')=='1':
    result=v5_generate_curriculum(project)
    print('Generation:',result['status'],'examples:',result.get('examples'))
if (project/V5_CURRICULUM_PATH).exists():
    result=v5_curriculum_checks(project)
    print('Verified curriculum:',result['status'])
    checks=result['checks']
    display(pd.DataFrame([{'Bounded goal':goal,'Training cases':checks['train_goals'].get(goal,0),'Development cases':checks['development_goals'].get(goal,0)} for goal in V5_GOALS]))
    provenance=result['provenance']
    print('Source games:',provenance['source_games'],'separate families:',provenance['family_count'],'external engine labels:',provenance['external_engine_labels'])
    print('Verified negative actions:',checks['train_negative_actions']+checks['development_negative_actions'])
    print('Dataset:',provenance['path'])
    print('SHA256:',provenance['sha256'])
    print('Terminal capture guard:',v5_validate_terminal_capture_guard(project)['status'])
else:
    print('Definitions validated. Generate the checkpointed curriculum with v5_generate_curriculum(project).')